# HPO-to-Organ Mapping Pipeline

## Purpose
Maps HPO phenotypes from gene module signatures to **anatomical organs** and **body systems** using shortest-path BFS traversal in the HPO ontology DAG, then computes per-organ gene percentages for each module.

---

## Pipeline Position
[hpo_structure_builder.ipynb] → hpo_to_organ_mapping.ipynb → [gganatomogram_Plot.Rmd]


---

## Input Files
| File | Description |
|------|-------------|
| `all_HPO_signature.csv` | Module-phenotype associations with gene lists (338 rows, 12 modules) |
| `combined_phenotype_map_*.csv` or `all_ird_hpo_map.csv` | HPO target IDs → organ/system mapping |
| `hp.obo` | HPO ontology for is_a traversal (~19,393 terms) |

---

## Output Files
| File | Description | Rows |
|------|-------------|------|
| `module_system_organ_summary_*.csv` | Complete mapping (module, label, score, genes) | 95 |
| `gganatogram_organs_input_*.csv` | Organ data for visualization (≥5% threshold) | 78 |
| `systems_input_*.csv` | Body system mapping data | variable |
| `audit_log_*.txt` | Pipeline execution log with statistics | - |

---

## Methodology Overview

### Step 1: Parse HPO Ontology
- Parse `hp.obo` and build is_a parent graph
- Skip obsolete terms (510 excluded)

### Step 2: Build Target Maps
- System targets: 139 HPO IDs → 4 body systems
- Organ targets: 221 HPO IDs → 57 organs

### Step 3: BFS Shortest-Path Cache
For each phenotype HPO ID:
1. If it's a target → return directly
2. Otherwise, BFS upward to find nearest target(s)
3. Return ALL targets at shortest distance

### Step 4: Per-Module Processing
For each module:
1. Get system/organ labels via cached BFS
2. Aggregate genes per label (set union)
3. Calculate percentage: (gene_count / module_size) × 100

### Step 5: Generate Outputs
- Combined summary with traceability
- gganatogram-ready CSV (filtered by threshold)
- Audit log with coverage statistics

---

## Key Parameters
| Parameter | Value | Description |
|-----------|-------|-------------|
| `MIN_PERCENT_TO_REPORT_ORGAN` | 5.0% | Minimum organ score for gganatogram output |
| `MIN_PERCENT_TO_REPORT_SYSTEM` | 0.0% | Minimum system score for output |
| `SKIP_OBSOLETE_TERMS` | True | Exclude obsolete HPO terms |

---

## Coverage Statistics
| Metric | Value |
|--------|-------|
| Unique HPO IDs in signatures | 299 |
| Mapped to organ targets | 259 (87%) |
| Mapped to system targets | 135 (45%) |
| Unmapped (organ) | 40 |
| Unmapped (system) | 164 |

---

## Notes
- BFS finds ALL targets at shortest distance (handles multi-parent terms)
- Gene aggregation uses set union (no double-counting)
- Supports legacy combined_phenotype_map format (auto-converts)
- Alternative mode: `all_IRD_HPO.csv` for all-IRD analysis without module separation

# HPO-to-Organ Mapping Pipeline

Maps HPO phenotypes from gene module signatures to anatomical organs using shortest-path BFS traversal in the HPO DAG, then computes per-organ gene percentages for each module.

**Inputs:**
- `all_HPO_signature.csv` - Module-phenotype associations with gene lists
- `combined_phenotype_map_*.csv` - HPO target IDs → organ/system mapping
- `hp.obo` - HPO ontology for is_a traversal

**Outputs:**
- Per-module organ summary CSV
- gganatogram input CSV
- Audit log

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict, deque
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cell 2: Configuration
SCRIPT_NAME = "hpo_to_organ_mapping"

# Paths (script runs from scripts/ folder)
INPUT_DIR = Path("../Input")
OUTPUT_DIR = Path(f"../output/{SCRIPT_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Input files

# HPO_SIGNATURE_FILE = INPUT_DIR / "all_IRD_HPO.csv"          # made for ALL IRD HPO's
HPO_SIGNATURE_FILE = INPUT_DIR / "all_HPO_signature.csv"

HPO_OBO_FILE = INPUT_DIR / "hp.obo"

# Find the most recent targets mapping file (new format)
target_map_candidates = []
target_map_dir = Path("../output/hpo_structure_builder")

candidate_patterns = [   # made for ALL IRD HPO's
    "all_ird_hpo_map.csv"
]

#candidate_patterns = [
 #   "phenotype_target_map_*.csv",
  #  "phenotype_targets_*.csv",
   # "target_map_*.csv",
    #"combined_phenotype_map_*.csv",
#]

for pattern in candidate_patterns:
    target_map_candidates.extend(target_map_dir.glob(pattern))

if target_map_candidates:
    TARGETS_FILE = max(target_map_candidates, key=lambda p: p.stat().st_mtime)
else:
    # Fall back to Input directory
    TARGETS_FILE = INPUT_DIR / "phenotype_targets.csv"
    if not TARGETS_FILE.exists():
        TARGETS_FILE = INPUT_DIR / "combined_phenotype_map.csv"

print(f"Using targets map: {TARGETS_FILE}")

# Pipeline configuration
MIN_PERCENT_TO_REPORT_ORGAN = 5.0  # Minimum organ percentage for gganatogram output
MIN_PERCENT_TO_REPORT_SYSTEM = 0.0  # Optional threshold for systems output
SKIP_OBSOLETE_TERMS = True
TOP_N_UNMAPPED = 20

def get_timestamp():
    """Generate timestamp for output files."""
    return datetime.now().strftime("%Y%m%d_%H%M")

TIMESTAMP = get_timestamp()
print(f"Output timestamp: {TIMESTAMP}")


In [ ]:
# Cell 3: Utility functions

def parse_gene_list(gene_string):
    """
    Parse comma-separated gene list string.
    Returns a set of gene symbols (case-sensitive, trimmed, no empty strings).
    """
    if pd.isna(gene_string) or gene_string == '':
        return set()
    genes = [g.strip() for g in str(gene_string).split(',')]
    return {g for g in genes if g}  # Filter out empty strings


def validate_hpo_id(hpo_id):
    """
    Validate HPO ID format (HP:ddddddd).
    """
    if pd.isna(hpo_id):
        return False
    return bool(re.match(r'^HP:\d{7}$', str(hpo_id)))

In [ ]:
# Cell 4: Parse HPO OBO file

def parse_hpo_obo(obo_path, skip_obsolete=True):
    """
    Parse hp.obo file and extract is_a relationships.
    Returns:
        - hpo_graph: dict mapping HPO ID -> list of parent HPO IDs (via is_a)
        - hpo_names: dict mapping HPO ID -> term name
    """
    hpo_graph = defaultdict(list)  # child -> [parents]
    hpo_names = {}
    obsolete_ids = set()

    current_id = None
    current_name = None
    current_parents = []
    current_is_obsolete = False
    in_term = False

    def commit_term():
        nonlocal current_id, current_name, current_parents, current_is_obsolete
        if not current_id:
            return
        if current_is_obsolete:
            obsolete_ids.add(current_id)
            return
        if current_name:
            hpo_names[current_id] = current_name
        if current_parents:
            hpo_graph[current_id].extend(current_parents)

    with open(obo_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if line == '[Term]':
                if in_term:
                    commit_term()
                in_term = True
                current_id = None
                current_name = None
                current_parents = []
                current_is_obsolete = False
                continue

            if line == '' or (line.startswith('[') and line != '[Term]'):
                if in_term:
                    commit_term()
                in_term = False
                continue

            if not in_term:
                continue

            if line.startswith('id: '):
                current_id = line[4:].strip()
            elif line.startswith('name: '):
                current_name = line[6:].strip()
            elif line.startswith('is_a: '):
                # Format: is_a: HP:0000001 ! All
                parent_part = line[6:].split('!')[0].strip()
                if parent_part.startswith('HP:'):
                    current_parents.append(parent_part)
            elif line.startswith('is_obsolete: '):
                if line.split(':', 1)[1].strip().lower() == 'true':
                    current_is_obsolete = True

    if in_term:
        commit_term()

    if skip_obsolete and obsolete_ids:
        # Remove edges pointing to obsolete terms
        for child in list(hpo_graph.keys()):
            hpo_graph[child] = [p for p in hpo_graph[child] if p not in obsolete_ids]

    print(f"Parsed HPO ontology: {len(hpo_names)} terms, {sum(len(v) for v in hpo_graph.values())} is_a edges")
    if skip_obsolete and obsolete_ids:
        print(f"  Skipped obsolete terms: {len(obsolete_ids)}")
    return dict(hpo_graph), hpo_names


# Parse the HPO ontology
hpo_graph, hpo_names = parse_hpo_obo(HPO_OBO_FILE, skip_obsolete=SKIP_OBSOLETE_TERMS)


In [ ]:
# Cell 5: Load and validate input files

# Load HPO signature data
df_signature = pd.read_csv(HPO_SIGNATURE_FILE)
print(f"Loaded signature data: {len(df_signature)} rows")
print(f"Columns: {list(df_signature.columns)}")
print(f"Modules: {sorted(df_signature['module_id'].unique())}")

# Validate required columns
required_cols = ['module_id', 'hpo_term_id', 'hpo_term_name',
                 'all_genes_in_module', 'genes_in_module_with_this_phenotype']
missing_cols = [c for c in required_cols if c not in df_signature.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if 'module_size' not in df_signature.columns:
    print("Note: 'module_size' column not found; using effective size from all_genes_in_module")

# Load targets mapping
df_targets = pd.read_csv(TARGETS_FILE)
print(f"\nLoaded targets mapping: {len(df_targets)} rows")
print(f"Columns: {list(df_targets.columns)}")

# Validate target columns
required_target_cols = ['target_hpo_id', 'target_type', 'label']
missing_target_cols = [c for c in required_target_cols if c not in df_targets.columns]

if missing_target_cols:
    legacy_has_hpo = 'hpo_id' in df_targets.columns
    legacy_has_organ = 'organ' in df_targets.columns
    legacy_has_system = 'system' in df_targets.columns

    if legacy_has_hpo and (legacy_has_organ or legacy_has_system):
        print("Detected legacy combined_phenotype_map format. Converting to new target map format...")
        rows = []
        for _, row in df_targets.iterrows():
            hpo_id = row.get('hpo_id')
            if pd.isna(hpo_id):
                continue
            hpo_id = str(hpo_id).strip()
            if not hpo_id:
                continue

            if legacy_has_organ:
                organ = row.get('organ')
                if pd.notna(organ) and str(organ).strip() != '':
                    organ_label = str(organ).strip()
                    rows.append({
                        'target_hpo_id': hpo_id,
                        'target_type': 'organ',
                        'label': organ_label,
                        'gganatogram_organ_name': organ_label,
                    })

            if legacy_has_system:
                system = row.get('system')
                if pd.notna(system) and str(system).strip() != '':
                    system_label = str(system).strip()
                    rows.append({
                        'target_hpo_id': hpo_id,
                        'target_type': 'system',
                        'label': system_label,
                    })

        if not rows:
            raise ValueError("Legacy target map detected, but no organ/system labels found to convert.")

        df_targets = pd.DataFrame(rows).drop_duplicates()
        print(f"Converted legacy mapping to {len(df_targets)} target rows")
        print(f"Columns after conversion: {list(df_targets.columns)}")
    else:
        raise ValueError(
            f"Missing required target columns: {missing_target_cols}. "
            f"Check that {TARGETS_FILE} uses the new target map format."
        )

# Re-check required columns after conversion
missing_target_cols = [c for c in required_target_cols if c not in df_targets.columns]
if missing_target_cols:
    raise ValueError(
        f"Missing required target columns: {missing_target_cols}. "
        f"Check that {TARGETS_FILE} uses the new target map format."
    )

# Normalize and validate target fields
df_targets['target_hpo_id'] = df_targets['target_hpo_id'].astype(str).str.strip()
df_targets['target_type'] = df_targets['target_type'].astype(str).str.strip().str.lower()
df_targets['label'] = df_targets['label'].astype(str).str.strip()

invalid_types = sorted(set(df_targets['target_type']) - {'system', 'organ'})
if invalid_types:
    raise ValueError(f"Invalid target_type values: {invalid_types}. Expected only 'system' or 'organ'.")

invalid_hpo_ids = [h for h in df_targets['target_hpo_id'] if not validate_hpo_id(h)]
if invalid_hpo_ids:
    sample = invalid_hpo_ids[:10]
    raise ValueError(f"Invalid target_hpo_id format for {len(invalid_hpo_ids)} rows. Sample: {sample}")

empty_labels = df_targets['label'].isna() | (df_targets['label'] == '')
if empty_labels.any():
    raise ValueError(f"Found {int(empty_labels.sum())} rows with empty label in targets mapping")

print("Target type counts:")
print(df_targets['target_type'].value_counts().to_string())

In [ ]:
# Cell 6: Build target HPO ID to labels mapping

def build_target_maps(df_targets):
    """
    Build separate mappings for system and organ targets.

    Returns:
        - system_target_to_labels: dict mapping target HPO ID -> set of system labels
        - organ_target_to_labels: dict mapping target HPO ID -> set of organ labels
        - organ_target_to_gganatogram: dict mapping target HPO ID -> set of gganatogram organ names
        - system_target_ids: set of system target HPO IDs
        - organ_target_ids: set of organ target HPO IDs
    """
    system_target_to_labels = defaultdict(set)
    organ_target_to_labels = defaultdict(set)
    organ_target_to_gganatogram = defaultdict(set)

    system_target_ids = set()
    organ_target_ids = set()

    for _, row in df_targets.iterrows():
        target_id = row['target_hpo_id']
        target_type = row['target_type']
        label = row['label']

        if target_type == 'system':
            system_target_ids.add(target_id)
            system_target_to_labels[target_id].add(label)
        elif target_type == 'organ':
            organ_target_ids.add(target_id)
            organ_target_to_labels[target_id].add(label)

            organ_name = row.get('gganatogram_organ_name')
            if pd.notna(organ_name) and str(organ_name).strip() != '':
                organ_target_to_gganatogram[target_id].add(str(organ_name).strip())
            else:
                organ_target_to_gganatogram[target_id].add(label)

    print(f"Built system targets: {len(system_target_ids)}")
    print(f"Built organ targets: {len(organ_target_ids)}")

    system_labels = set()
    for labels in system_target_to_labels.values():
        system_labels.update(labels)
    organ_labels = set()
    for labels in organ_target_to_labels.values():
        organ_labels.update(labels)

    print(f"Unique system labels: {len(system_labels)}")
    print(f"Unique organ labels: {len(organ_labels)}")

    return (
        dict(system_target_to_labels),
        dict(organ_target_to_labels),
        dict(organ_target_to_gganatogram),
        system_target_ids,
        organ_target_ids,
    )


(
    system_target_to_labels,
    organ_target_to_labels,
    organ_target_to_gganatogram,
    SYSTEM_TARGET_IDS,
    ORGAN_TARGET_IDS,
) = build_target_maps(df_targets)

print("\nSample system mappings:")
for hpo_id in list(SYSTEM_TARGET_IDS)[:5]:
    print(f"  {hpo_id}: {system_target_to_labels.get(hpo_id, set())}")

print("\nSample organ mappings:")
for hpo_id in list(ORGAN_TARGET_IDS)[:5]:
    print(f"  {hpo_id}: {organ_target_to_labels.get(hpo_id, set())}")

In [ ]:
# Cell 7: Row deduplication

def deduplicate_signature_rows(df):
    """
    Deduplicate by (module_id, hpo_term_id).
    For duplicates, take union of genes_in_module_with_this_phenotype.
    Returns deduplicated dataframe and list of warnings.
    """
    warnings_list = []

    # Group by module_id and hpo_term_id
    grouped = df.groupby(['module_id', 'hpo_term_id'])

    dedup_rows = []
    for (mod_id, hpo_id), group in grouped:
        if len(group) > 1:
            warnings_list.append(f"Duplicate (module={mod_id}, hpo={hpo_id}): {len(group)} rows merged")

            # Merge gene lists
            all_genes = set()
            for genes_str in group['genes_in_module_with_this_phenotype']:
                all_genes.update(parse_gene_list(genes_str))

            # Take first row and update genes
            row = group.iloc[0].copy()
            row['genes_in_module_with_this_phenotype'] = ','.join(sorted(all_genes))
            dedup_rows.append(row)
        else:
            dedup_rows.append(group.iloc[0])

    df_dedup = pd.DataFrame(dedup_rows)
    print(f"Deduplication: {len(df)} -> {len(df_dedup)} rows")
    if warnings_list:
        print(f"  Merged {len(warnings_list)} duplicate groups")

    return df_dedup, warnings_list


df_signature_dedup, dedup_warnings = deduplicate_signature_rows(df_signature)

In [ ]:
# Cell 8: BFS shortest-path cache builder

def bfs_find_nearest_targets(hpo_id, hpo_graph, target_ids):
    """
    BFS upward through is_a edges to find the nearest target HPO ID(s).
    Returns set of target HPO IDs at the shortest distance, or empty set if unreachable.
    """
    if hpo_id in target_ids:
        return {hpo_id}

    visited = {hpo_id}
    queue = deque([(hpo_id, 0)])
    found_targets = set()
    shortest_dist = float('inf')

    while queue:
        current, dist = queue.popleft()

        # If we've found targets and current distance exceeds shortest, stop
        if found_targets and dist > shortest_dist:
            break

        # Get parents (is_a relationships)
        parents = hpo_graph.get(current, [])

        for parent in parents:
            if parent in visited:
                continue
            visited.add(parent)

            if parent in target_ids:
                found_targets.add(parent)
                shortest_dist = dist + 1
            else:
                queue.append((parent, dist + 1))

    return found_targets


def build_hpo_to_targets_cache(unique_hpo_ids, hpo_graph, target_ids, cache_name):
    """
    Build cache mapping each unique HPO ID to its nearest target HPO IDs.
    """
    cache = {}
    unmapped = []

    for hpo_id in unique_hpo_ids:
        targets = bfs_find_nearest_targets(hpo_id, hpo_graph, target_ids)
        if targets:
            cache[hpo_id] = targets
        else:
            unmapped.append(hpo_id)
            cache[hpo_id] = set()

    print(f"Built {cache_name} cache: {len(cache)} entries")
    print(f"  Mapped to targets: {len(cache) - len(unmapped)}")
    print(f"  Unmapped (no path to targets): {len(unmapped)}")

    return cache, unmapped


# Get unique HPO IDs from signature data
unique_hpo_ids = set(df_signature_dedup['hpo_term_id'].dropna().unique())
print(f"Unique HPO IDs in signature: {len(unique_hpo_ids)}")

# Validate HPO IDs against ontology
missing_in_ontology = [h for h in unique_hpo_ids if h not in hpo_names]
if missing_in_ontology:
    print(f"Warning: {len(missing_in_ontology)} HPO IDs not found in ontology")

# Build independent caches
hpo_to_system_targets_cache, unmapped_system_hpo_ids = build_hpo_to_targets_cache(
    unique_hpo_ids, hpo_graph, SYSTEM_TARGET_IDS, cache_name='system'
)
hpo_to_organ_targets_cache, unmapped_organ_hpo_ids = build_hpo_to_targets_cache(
    unique_hpo_ids, hpo_graph, ORGAN_TARGET_IDS, cache_name='organ'
)

# Mapping coverage stats
mapped_to_system = {h for h, t in hpo_to_system_targets_cache.items() if t}
mapped_to_organ = {h for h, t in hpo_to_organ_targets_cache.items() if t}

system_only = mapped_to_system - mapped_to_organ
organ_only = mapped_to_organ - mapped_to_system


In [ ]:
# Cell 9: Per-module label assignment (system and organ)

def compute_module_size_effective(df_module):
    """
    Compute effective module size from union of all_genes_in_module.
    Returns (size, genes_set).
    """
    all_module_genes = set()
    for _, row in df_module.iterrows():
        module_genes = parse_gene_list(row['all_genes_in_module'])
        all_module_genes.update(module_genes)
    return len(all_module_genes), all_module_genes


def compute_module_label_summary(df_module, hpo_to_targets_cache, target_to_labels, hpo_names_lookup):
    """
    Compute label summary for a single module.

    Returns dict with:
        - label_data: dict mapping label -> {genes, hpo_ids, hpo_names, target_ids}
        - unmapped_phenotypes: list of HPO IDs that couldn't be mapped
    """
    label_data = defaultdict(lambda: {
        'genes': set(),
        'hpo_ids': set(),
        'hpo_names': set(),
        'target_ids': set()
    })

    unmapped_phenotypes = []

    for _, row in df_module.iterrows():
        hpo_id = row['hpo_term_id']
        hpo_name = row['hpo_term_name']
        genes = parse_gene_list(row['genes_in_module_with_this_phenotype'])

        # Get target HPO IDs for this phenotype
        target_hpo_ids = hpo_to_targets_cache.get(hpo_id, set())

        if not target_hpo_ids:
            unmapped_phenotypes.append(hpo_id)
            continue

        # Map to labels through targets
        for target_id in target_hpo_ids:
            labels = target_to_labels.get(target_id, set())
            for label in labels:
                label_data[label]['genes'].update(genes)
                label_data[label]['hpo_ids'].add(hpo_id)
                label_data[label]['hpo_names'].add(
                    hpo_name if pd.notna(hpo_name) else hpo_names_lookup.get(hpo_id, hpo_id)
                )
                label_data[label]['target_ids'].add(target_id)

    return dict(label_data), unmapped_phenotypes


def process_all_modules(df_dedup,
                        hpo_to_system_targets_cache,
                        hpo_to_organ_targets_cache,
                        system_target_to_labels,
                        organ_target_to_labels,
                        organ_target_to_gganatogram,
                        hpo_names_lookup):
    """
    Process all modules and generate system/organ summary data.
    """
    all_results = []
    module_audit = []
    gganatogram_rows = []

    for module_id in sorted(df_dedup['module_id'].unique()):
        df_module = df_dedup[df_dedup['module_id'] == module_id]

        # Compute effective module size
        module_size_effective, _ = compute_module_size_effective(df_module)

        # Optional consistency check if module_size is present
        if 'module_size' in df_module.columns:
            declared_size = df_module['module_size'].iloc[0]
            if declared_size != module_size_effective:
                module_audit.append(
                    f"Module {module_id}: declared_size={declared_size}, effective_size={module_size_effective}"
                )

        # Compute summaries independently
        system_data, system_unmapped = compute_module_label_summary(
            df_module, hpo_to_system_targets_cache, system_target_to_labels, hpo_names_lookup
        )
        organ_data, organ_unmapped = compute_module_label_summary(
            df_module, hpo_to_organ_targets_cache, organ_target_to_labels, hpo_names_lookup
        )
        organ_gg_data, _ = compute_module_label_summary(
            df_module, hpo_to_organ_targets_cache, organ_target_to_gganatogram, hpo_names_lookup
        )

        if system_unmapped or organ_unmapped:
            module_audit.append(
                f"Module {module_id}: system_unmapped={len(system_unmapped)}, organ_unmapped={len(organ_unmapped)}"
            )

        # Generate combined result rows
        for label, data in system_data.items():
            gene_count = len(data['genes'])
            score_percent = round(100.0 * gene_count / module_size_effective, 1) if module_size_effective > 0 else 0.0

            all_results.append({
                'module_id': module_id,
                'module_size_effective': module_size_effective,
                'label_type': 'system',
                'label': label,
                'gene_count': gene_count,
                'score_percent': score_percent,
                'contributing_hpo_ids': ';'.join(sorted(data['hpo_ids'])),
                'contributing_hpo_names': ';'.join(sorted(data['hpo_names'])),
                'matched_target_hpo_ids': ';'.join(sorted(data['target_ids'])),
                'genes_assigned': ','.join(sorted(data['genes']))
            })

        for label, data in organ_data.items():
            gene_count = len(data['genes'])
            score_percent = round(100.0 * gene_count / module_size_effective, 1) if module_size_effective > 0 else 0.0

            all_results.append({
                'module_id': module_id,
                'module_size_effective': module_size_effective,
                'label_type': 'organ',
                'label': label,
                'gene_count': gene_count,
                'score_percent': score_percent,
                'contributing_hpo_ids': ';'.join(sorted(data['hpo_ids'])),
                'contributing_hpo_names': ';'.join(sorted(data['hpo_names'])),
                'matched_target_hpo_ids': ';'.join(sorted(data['target_ids'])),
                'genes_assigned': ','.join(sorted(data['genes']))
            })

        # gganatogram input (organ names only)
        for organ_name, data in organ_gg_data.items():
            gene_count = len(data['genes'])
            score_percent = round(100.0 * gene_count / module_size_effective, 1) if module_size_effective > 0 else 0.0
            gganatogram_rows.append({
                'module_id': module_id,
                'organ': organ_name,
                'value': score_percent
            })

    result_columns = [
        'module_id', 'module_size_effective', 'label_type', 'label', 'gene_count', 'score_percent',
        'contributing_hpo_ids', 'contributing_hpo_names', 'matched_target_hpo_ids', 'genes_assigned'
    ]
    gganatogram_columns = ['module_id', 'organ', 'value']

    df_results = pd.DataFrame(all_results, columns=result_columns)
    df_gganatogram = pd.DataFrame(gganatogram_rows, columns=gganatogram_columns)

    return df_results, df_gganatogram, module_audit


# Process all modules
df_results, df_gganatogram, audit_messages = process_all_modules(
    df_signature_dedup,
    hpo_to_system_targets_cache,
    hpo_to_organ_targets_cache,
    system_target_to_labels,
    organ_target_to_labels,
    organ_target_to_gganatogram,
    hpo_names
)

print(f"\nGenerated {len(df_results)} module-label rows")
if df_results.empty:
    print("No labels mapped. Check target lists, HPO IDs, and ontology coverage.")
else:
    print(f"Modules processed: {df_results['module_id'].nunique()}")
    print(f"Unique system labels: {df_results[df_results['label_type'] == 'system']['label'].nunique()}")
    print(f"Unique organ labels: {df_results[df_results['label_type'] == 'organ']['label'].nunique()}")

In [ ]:
# Cell 10: Generate outputs

# Output 1: Combined system + organ summary
output_cols = [
    'module_id', 'module_size_effective', 'label_type', 'label', 'gene_count', 'score_percent',
    'contributing_hpo_ids', 'contributing_hpo_names', 'matched_target_hpo_ids'
]
if 'genes_assigned' in df_results.columns:
    output_cols.append('genes_assigned')

df_output = df_results[output_cols].sort_values(
    ['module_id', 'label_type', 'score_percent'], ascending=[True, True, False]
)

output_file_1 = OUTPUT_DIR / f"module_system_organ_summary_{TIMESTAMP}.csv"
df_output.to_csv(output_file_1, index=False)
print(f"Saved: {output_file_1}")

# Output 2: gganatogram input (organs only, filtered by minimum threshold)
df_gganatogram_filtered = df_gganatogram.copy()
if MIN_PERCENT_TO_REPORT_ORGAN > 0:
    df_gganatogram_filtered = df_gganatogram_filtered[
        df_gganatogram_filtered['value'] >= MIN_PERCENT_TO_REPORT_ORGAN
    ]

output_file_2 = OUTPUT_DIR / f"gganatogram_organs_input_{TIMESTAMP}.csv"
df_gganatogram_filtered.to_csv(output_file_2, index=False)
print(f"Saved: {output_file_2}")
print(f"  Rows with value >= {MIN_PERCENT_TO_REPORT_ORGAN}%: {len(df_gganatogram_filtered)}")

# Output 3: systems input (systems only)
df_systems_input = df_results[df_results['label_type'] == 'system'][
    ['module_id', 'label', 'score_percent']
].copy()
df_systems_input = df_systems_input.rename(columns={'label': 'system', 'score_percent': 'value'})
if MIN_PERCENT_TO_REPORT_SYSTEM > 0:
    df_systems_input = df_systems_input[df_systems_input['value'] >= MIN_PERCENT_TO_REPORT_SYSTEM]

output_file_3 = OUTPUT_DIR / f"systems_input_{TIMESTAMP}.csv"
df_systems_input.to_csv(output_file_3, index=False)
print(f"Saved: {output_file_3}")


In [ ]:
# Cell 11: Audit log summary

audit_file = OUTPUT_DIR / f"audit_log_{TIMESTAMP}.txt"

with open(audit_file, 'w') as f:
    f.write("HPO-to-System/Organ Mapping Pipeline Audit Log\n")
    f.write(f"Generated: {datetime.now().isoformat()}\n")
    f.write("="*60 + "\n\n")

    f.write("INPUT FILES:\n")
    f.write(f"  Signature: {HPO_SIGNATURE_FILE}\n")
    f.write(f"  Targets map: {TARGETS_FILE}\n")
    f.write(f"  HPO ontology: {HPO_OBO_FILE}\n\n")

    f.write("CONFIGURATION:\n")
    f.write(f"  MIN_PERCENT_TO_REPORT_ORGAN: {MIN_PERCENT_TO_REPORT_ORGAN}%\n")
    f.write(f"  MIN_PERCENT_TO_REPORT_SYSTEM: {MIN_PERCENT_TO_REPORT_SYSTEM}%\n")
    f.write(f"  SKIP_OBSOLETE_TERMS: {SKIP_OBSOLETE_TERMS}\n")
    f.write(f"  TOP_N_UNMAPPED: {TOP_N_UNMAPPED}\n\n")

    f.write("HPO ONTOLOGY:\n")
    f.write(f"  Terms loaded: {len(hpo_names)}\n")
    f.write(f"  System target HPO IDs: {len(SYSTEM_TARGET_IDS)}\n")
    f.write(f"  Organ target HPO IDs: {len(ORGAN_TARGET_IDS)}\n\n")

    f.write("SIGNATURE DATA:\n")
    f.write(f"  Input rows: {len(df_signature)}\n")
    f.write(f"  After dedup: {len(df_signature_dedup)}\n")
    f.write(f"  Unique HPO IDs: {len(unique_hpo_ids)}\n")
    f.write(f"  Unmapped (system): {len(unmapped_system_hpo_ids)}\n")
    f.write(f"  Unmapped (organ): {len(unmapped_organ_hpo_ids)}\n\n")

    f.write("MAPPING COVERAGE:\n")
    f.write(f"  Phenotypes mapped to system: {len(mapped_to_system)}\n")
    f.write(f"  Phenotypes mapped to organ: {len(mapped_to_organ)}\n")
    f.write(f"  System-only phenotypes: {len(system_only)}\n")
    f.write(f"  Organ-only phenotypes: {len(organ_only)}\n\n")

    f.write("OUTPUT SUMMARY:\n")
    f.write(f"  Total module-label rows: {len(df_results)}\n")
    f.write(f"  Modules: {df_results['module_id'].nunique()}\n")
    f.write(f"  Unique system labels: {df_results[df_results['label_type'] == 'system']['label'].nunique()}\n")
    f.write(f"  Unique organ labels: {df_results[df_results['label_type'] == 'organ']['label'].nunique()}\n")
    f.write(f"  gganatogram rows (>={MIN_PERCENT_TO_REPORT_ORGAN}%): {len(df_gganatogram_filtered)}\n")
    f.write(f"  systems rows (>={MIN_PERCENT_TO_REPORT_SYSTEM}%): {len(df_systems_input)}\n\n")

    if dedup_warnings:
        f.write("DEDUPLICATION WARNINGS:\n")
        for w in dedup_warnings:
            f.write(f"  {w}\n")
        f.write("\n")

    if audit_messages:
        f.write("MODULE AUDIT:\n")
        for m in audit_messages:
            f.write(f"  {m}\n")
        f.write("\n")

    if unmapped_system_hpo_ids:
        f.write("TOP UNMAPPED HPO IDs (system):\n")
        for hpo_id in sorted(unmapped_system_hpo_ids)[:TOP_N_UNMAPPED]:
            name = hpo_names.get(hpo_id, 'UNKNOWN')
            f.write(f"  {hpo_id}: {name}\n")
        if len(unmapped_system_hpo_ids) > TOP_N_UNMAPPED:
            f.write(f"  ... and {len(unmapped_system_hpo_ids) - TOP_N_UNMAPPED} more\n")
        f.write("\n")

    if unmapped_organ_hpo_ids:
        f.write("TOP UNMAPPED HPO IDs (organ):\n")
        for hpo_id in sorted(unmapped_organ_hpo_ids)[:TOP_N_UNMAPPED]:
            name = hpo_names.get(hpo_id, 'UNKNOWN')
            f.write(f"  {hpo_id}: {name}\n")
        if len(unmapped_organ_hpo_ids) > TOP_N_UNMAPPED:
            f.write(f"  ... and {len(unmapped_organ_hpo_ids) - TOP_N_UNMAPPED} more\n")
        f.write("\n")

print(f"Saved: {audit_file}")
print("\n" + "="*60)
print("Pipeline completed successfully!")
print("="*60)

In [ ]:
# Cell 12: Preview results

print("Top labels by score (first 3 modules):")
for mod_id in sorted(df_results['module_id'].unique())[:3]:
    print(f"\nModule {mod_id}:")
    for label_type in ['system', 'organ']:
        subset = df_results[df_results['module_id'] == mod_id]
        subset = subset[subset['label_type'] == label_type]
        if subset.empty:
            continue
        print(f"  {label_type.capitalize()}:")
        top_rows = subset.nlargest(5, 'score_percent')
        for _, row in top_rows.iterrows():
            print(f"    {row['label']:30s} {row['score_percent']:6.1f}% ({row['gene_count']} genes)")